# Frozen decoder-context parity on NVIDIA L4

Select an **L4** runtime and run all cells. This replays one P:0006 direction, **3T → 7T T1w**, against the archived matched-calibration pilot. The wrong translator request is 0.1T T1w. All transport arms use the 7T decoder and renderer.

A pass requires an exact solver and tensor trace, all 12 raw and calibrated image fingerprints, and all 36 scores within an absolute tolerance of `1e-10`. Each launch writes a fresh `run-NNNN/summary.json` under `MyDrive/MRIxFields2026/UnifiedStage2_1ca2b4a_01/stage2_decoder_context_parity/b6d9ab4b2971b14ba2a991d28dad91c4430503f3/`.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys
from tempfile import TemporaryDirectory

from google.colab import drive

gpu = subprocess.check_output(
    ["nvidia-smi", "--id=0", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip()
if "L4" not in gpu:
    raise RuntimeError(f"Select the NVIDIA L4 runtime used for the archived pilot. Assigned GPU: {gpu}")
drive.mount("/content/drive")

commit = "b6d9ab4b2971b14ba2a991d28dad91c4430503f3"
remote = "https://github.com/GuillermoTafoya/MRIxFields.git"
with TemporaryDirectory(prefix="mrix-parity-code-", dir="/content") as checkout:
    repo = Path(checkout)
    def git(*args):
        return subprocess.check_output(["git", "-C", checkout, *args], text=True).strip()

    git("init", "--quiet")
    git("fetch", "--quiet", "--depth=1", remote, commit)
    git("checkout", "--quiet", "--detach", "FETCH_HEAD")
    if git("rev-parse", "HEAD") != commit:
        raise RuntimeError("The implementation commit pin did not verify.")
    env = dict(os.environ, FIELDBRIDGE_IMPLEMENTATION_COMMIT=commit,
               CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONDONTWRITEBYTECODE="1", PYTHONUNBUFFERED="1",
               PYTHONPATH=os.pathsep.join([str(repo / "src"), str(repo / "notebooks")]))
    subprocess.run([sys.executable, "-B", str(repo / "notebooks/stage2_decoder_context_parity_operator.py")],
                   cwd=repo, env=env, check=True)